# 01 数据质量检查

本 Notebook 用于复查阶段 1 和阶段 2.5 的数据链路：AKShare 日频行情缓存、全量下载 manifest、失败列表、全市场质量报告，以及基于全量缓存重建后的阶段 2 输出。

## 当前数据状态

- 股票列表来源：`config/data.yml` 中的 `symbol_list_endpoint: spot_sina`。
- 行情端点：`endpoint: daily_sina`，复权口径：`adjust: qfq`。
- 全量列表数量：5207。
- 已缓存标准化行情：5206 只。
- 下载失败：1 只，`688797`，AKShare 返回缺少 `date` 字段。
- 全量 daily panel 行数：10,674,838。
- 月度股票池行数：188,467。
- 标签行数：184,246。

注意：这里的“全量”指 AKShare 当前可获取股票列表中的全量日频数据，不等价于严格无幸存者偏差的历史全市场。

## 清洗原则

- 原始 AKShare 返回和标准化行情分层保存。
- 只做字段标准化、日期解析、数值类型转换、成交量单位统一和排序。
- 不前向填充 OHLCV，不伪造停牌交易日，不自动删除异常涨跌幅样本。
- `daily_sina` 成交量按“股”处理，不再乘以 100。
- `vwap = amount / volume`，但前复权 OHLC 与原始成交额/成交量口径可能不一致，因此 `vwap_outside_bar_count` 只作为提示。
- 跳过已有缓存前，必须确认标准化文件中的 `source` 和 `adjust` 与本次配置一致。

In [ ]:
from io import StringIO

import pandas as pd

from quant_rl_alpha.utils.config import load_config
from quant_rl_alpha.utils.paths import project_root

root = project_root()
data_config = load_config("data")
data_config

## 输出文件检查

先确认全量下载和阶段 2 重建产物是否存在，并查看文件大小。

In [ ]:
path_items = {
    "symbols": "data/reports/full_a_stock_symbols.csv",
    "manifest": "data/reports/full_download_manifest.csv",
    "failures": "data/reports/full_download_failures.csv",
    "quality": "data/reports/full_data_quality.md",
    "summary": "data/reports/full_data_ingestion.md",
    "daily_panel": "data/interim/daily_panel.parquet",
    "universe": "data/processed/universe_monthly.parquet",
    "labels": "data/processed/labels.parquet",
}
paths = {name: root / rel_path for name, rel_path in path_items.items()}
file_rows = []
for name, path in paths.items():
    size_mb = round(path.stat().st_size / 1024 / 1024, 2) if path.exists() else 0
    file_rows.append({"name": name, "path": str(path), "exists": path.exists(), "size_mb": size_mb})
pd.DataFrame(file_rows)

## 下载 Manifest

`manifest` 是全量下载的主审计表。重点看状态分布、行数、首末日期、端点和复权口径。

In [ ]:
manifest = pd.read_csv(paths["manifest"])

display(manifest["status"].value_counts().rename("count"))
display(manifest.head())

In [ ]:
pd.DataFrame([
    {
        "symbols": len(manifest),
        "standard_rows": int(manifest["standard_rows"].sum()),
        "raw_rows": int(manifest["raw_rows"].sum()),
        "first_date_min": manifest["first_date"].dropna().min(),
        "last_date_max": manifest["last_date"].dropna().max(),
        "failed": int((manifest["status"] == "failed").sum()),
    }
])

## 失败列表

失败股票不应静默跳过。它们应保留在失败列表里，后续可以单独重试或人工判断是否应从当前研究样本中排除。

In [ ]:
failures = pd.read_csv(paths["failures"])
failures

## 质量报告

质量报告是 Markdown 文件，明细部分嵌入 CSV。下面把 CSV 读回 DataFrame，方便排序和汇总。

In [ ]:
quality_text = paths["quality"].read_text(encoding="utf-8")
quality_csv = quality_text.split("```csv", 1)[1].split("```", 1)[0].strip()
quality = pd.read_csv(StringIO(quality_csv))

quality.shape

In [ ]:
issue_cols = (
    "duplicate_date_count missing_required_count missing_ohlcv_count non_positive_price_count "
    "zero_volume_count negative_volume_count negative_amount_count volume_amount_mismatch_count "
    "ohlc_inconsistent_count large_return_count vwap_outside_bar_count"
).split()

quality[issue_cols].sum().sort_values(ascending=False).to_frame("total")

In [ ]:
top_vwap_cols = "symbol name rows start_date end_date vwap_outside_bar_count has_issue".split()
quality.sort_values("vwap_outside_bar_count", ascending=False).head(10)[top_vwap_cols]

质量报告中大面积 `has_issue=True` 主要来自 `vwap_outside_bar_count`。这与当前口径一致：前复权 OHLC 和原始成交额/成交量计算出的 VWAP 可能不在同一复权尺度上。第一版只记录这个风险，不自动修正，也不据此删除样本。

## 阶段 2 输出复查

全量下载完成后，`stage2-build` 已基于标准化缓存重建 daily panel、月度股票池和标签。下面只读取元信息，避免在 Notebook 中做不必要的大规模展示。

In [ ]:
stage2_rows = []
for name in ["daily_panel", "universe", "labels"]:
    frame = pd.read_parquet(paths[name])
    start_date = frame["date"].min() if "date" in frame else None
    end_date = frame["date"].max() if "date" in frame else None
    row = {"name": name, "rows": len(frame), "columns": list(frame.columns)}
    stage2_rows.append(row | {"start_date": start_date, "end_date": end_date})

pd.DataFrame(stage2_rows)

## 后续建议

- 后续小规模实验不要删除全量缓存，应通过配置选择子集。
- `688797` 可以后续单独重试；当前失败原因已经进入失败列表。
- 在进入回测前，需要继续显式处理停牌、涨跌停、开盘价缺失、买卖失败和交易成本。
- 当前股票列表来自 AKShare 当前可得列表，不能当成无幸存者偏差的历史全市场。